# PopGen Pipline Platform (PPP) Analysis Notebook

## Generating Prerequisite Files

The required input files for a PPP run are:
- A genome VCF of the target populations (plus a tabix index if bgzipped)
- A population model file

The population model file is a JSON-formatted file that defines population names and the individuals from the VCF that belong to each population. This file can be created using the model_creator function, or by creating it manually by using an example model file as a template. 

For region filtering, the following should be provided:
- Name for target region file
- Name for final selected region file
- File with genic regions/regions to be excluded from analysis (optional)
- File with regions to be selected from for analysis (optional)

There are two methods for obtaining target regions for subsampling and analysis, with some level of interoperability: using a statistic file to sample regions, or use a file with target regions to randomly select regions with enough sites to be valid in an IM analysis. To generate a statistic file, use vcf_calc to read over the input VCF file, then use stat_sampler to select either a random or uniform distribution of these regions given their statistic value. These stat files can be filtered with a genic region file, available from UCSC, using the subtract_bed function. This method is implemented in this notebook, with the genic region filtering offered as an optional cell.

An additional method of region selection, done without statistics, can be done by downloading region files for genic regions, and optionally for STR regions and regions with missing data. If one wants to find a set of target regions that are intergenic and outside of STR regions, download the corresponding files from UCSC for your species and use invert_bed_region to 'invert' the files, optionally selecting only regions outside of a set number of bases from the regions in the file with the --window option. If files don't have typical BED column order, use the --bed-column-index option to provide a comma-separated string with the 0-based index of the start, end, and chromosome column. (For normal BED files, this would be '1,2,0') The get_nonmissing_chunks function can scan the input VCF and find regions with no missing data. All of these can be combined with bedtools to select regions that overlap with all three possible files. 

Whichever method used, the target region file should be run through the informative_filter function to check the VCF file has enough biallelic, informative SNPs (two or more of each of two alleles) to have a good chance of passing the four-gamete test (which requires regions with at least two SNPs). An informative count of 5 will usually allow for this, but if you have limited regions this threshhold can be lowered to 3. 


In addition to these files, additional functionality such as CpG filtering and comprehensive logging can be used by providing:
- A reference FASTA (for CpG filtering, can be bgzipped or unzipped but requires indexing w/faidx)
- A log filename

This section also is used to set up the directory structure for data files, a working directory, a directory for VCF region files, a target number of loci, and names for various stages of loci VCF files (phasing, four-gamete testing, potentially filtering individuals with missing data).

## PPP Libraries

Below we will import the necessary tools and libraries from the py-popgen environment to carry out our analyses

In [1]:
from datetime import datetime
import sys
import os
import subprocess
from pgpipe import vcf_split_pysam, vcf_to_ima, vcf_filter, vcf_calc, vcf_phase, stat_sampler, vcf_split
from pgpipe.logging_module import initLogger
from pgpipe.informative_loci_filter import filter_bed_regions
from pgpipe.subtract_bed import filter_stat
from pgpipe import admixture, graph_plotter
import pandas as pd
import pysam

print ("Imports complete")

Imports complete


## PPP Parameters

Below we define a set of parameters utilized by PPP core functions. Default fallback values are predefined however, actual values set by the user in the `config.yaml` file will override these defaults. 

Note that the following code cell has the cell tag `parameters`. This allows papermill to inject the values from the `config.yaml` file into the notebook.


In [2]:
try:
    organsim = snakemake.params.organism
    data_dir = snakemake.params.data_dir
    main_vcf_name = snakemake.params.main_vcf_name
    filtered_vcf_name = snakemake.params.filtered_vcf_name
    int_bed_file = snakemake.params.int_bed_file
    target_loci_file = snakemake.params.target_loci_file
    stat_file_name = snakemake.params.stat_file_name
    tajimaD_file = snakemake.params.tajimaD_file
    windowed_pi_file = snakemake.params.windowed_pi_file
    loci = snakemake.params.loci
    model = snakemake.params.model
    model_file = snakemake.params.model_file
    statistic_window_size = snakemake.params.statistic_window_size
    statistic_window_step = snakemake.params.statistic_window_step

except NameError:
    pass



In [4]:
time = datetime.today().strftime('%Y-%m-%d %H:%M:%S')
print(f"PPP Notebook executed at: {time}")

PPP Notebook executed at: 2026-10-22 19:21:35


## Filtering with `vcf_filter`

The vcf_filter step will filter the original VCF according to many conditions, including:
- Non-biallelic sites (--filter-min-alleles and --filter-max-alleles)
- Sites with missing data (--filter-max-missing)
- Indels 
- Sites on non-autosomal chromosomes (--filter-exclude-chr)
- Individuals not named in model file (use --model-file to input model file, with --model if multiple models included in file)


In [ ]:

vcf_filter.run(
    vcf=main_vcf_name,
    model_file=model_file,
    model=f"{model}", 
    filter_min_alleles=2, 
    filter_max_alleles=2, 
    out_format='vcf.gz', 
    out=filtered_vcf_name, 
    overwrite=True
)

pysam.tabix_index(filtered_vcf_name, preset='vcf', force=True)
print("Filtering complete")

Filtering complete


## Population Genetics Statistics with `vcf_calc`

The vcf_calc step will, for every 10kb window in the genome, calculate Fst given populations from the model file. Statistics that can be filtered over currently include:
- Windowed pi
- Tajima's D
- Fst

### Windowed Pi

In [12]:
vcf_calc.run(
    vcf=filtered_vcf_name,
    out=windowed_pi_file,
    calc_statistic='window-pi',
    model=f"{model}",
    statistic_window_size=statistic_window_size,
    overwrite=True
)
print("Windowed PI Stat calculation complete")
print(pd.read_csv(windowed_pi_file, sep='\t', header=None, nrows=5))

Windowed PI Stat calculation complete
       0          1        2           3            4
0  CHROM  BIN_START  BIN_END  N_VARIANTS           PI
1   chr1    1030001  1040000           3  0.000107487
2   chr1    1040001  1050000           6  0.000129055
3   chr1    1050001  1060000           7   8.3066e-05
4   chr1    1060001  1070000           1  2.99465e-05


### Fst

In [13]:
vcf_calc.run(
    vcf=filtered_vcf_name,
    out=stat_file_name,
    calc_statistic='windowed-weir-fst',
    model=f"{model}",
    statistic_window_size=statistic_window_size,
    statistic_window_step=statistic_window_step,
    model_file=model_file,
    overwrite=True
)
print("Fst Stat calculation complete")
print(pd.read_csv(stat_file_name, sep='\t', header=None, nrows=5))

Fst Stat calculation complete
       0          1        2           3             4         5
0  CHROM  BIN_START  BIN_END  N_VARIANTS  WEIGHTED_FST  MEAN_FST
1   chr1    1030001  1040000           3      0.362861   0.18178
2   chr1    1040001  1050000           6      0.843137  0.472386
3   chr1    1050001  1060000           7      0.679709  0.122125
4   chr1    1060001  1070000           1      0.122186  0.122186


### Tajima's D

In [14]:
vcf_calc.run(
    vcf=filtered_vcf_name,
    out=tajimaD_file,
    calc_statistic='TajimaD',
    model=f"{model}",
    statistic_window_size=statistic_window_size,
    overwrite=True
)
print("TajimaD Stat calculation complete")
print(pd.read_csv(tajimaD_file, sep='\t', header=None, nrows=5))

TajimaD Stat calculation complete
       0          1       2          3
0  CHROM  BIN_START  N_SNPS    TajimaD
1   chr1    1030000       3    1.05714
2   chr1    1040000       6  -0.333732
3   chr1    1050000       7   -1.47728
4   chr1    1060000       1   0.336267
